# Traduction betsileo ↔ malgache officiel : ByT5-small sur Kaggle

Même dataset, mêmes découpages, mêmes métriques et mêmes garde-fous que le notebook NLLB : les deux résultats
(bloc 10) se comparent directement dans le mémoire. Lance-le de haut en bas, ou *Save Version → Save & Run All (Commit)*.

**Avant de lancer**
1. *Settings → Accelerator* : **GPU T4 x2**. Ce notebook utilise les **deux** GPU (voir le bloc 2).
2. *Settings → Internet* : **On**.
3. *Add-ons → Secrets* : `HF_TOKEN` (droit **write**), activé pour ce notebook.
4. Données : dataset Kaggle avec `train.jsonl`, `validation.jsonl`, `test.jsonl` (*Add Input*), sinon `DATASET_REPO`.

**ByT5 face à NLLB**

| | ByT5-small | NLLB-200 600M |
|---|---|---|
| Lecture | octet par octet : pas de vocabulaire, aucun mot inconnu, robuste aux fautes de frappe et aux variantes d'orthographe (vonona/vognona, n/gn) | morceaux de mots |
| Malgache | vu au pré-entraînement (mC4, 101 langues dont le malgache) : il connaît le malgache écrit, mais **n'a jamais appris à traduire** | pré-entraîné à **traduire** le malgache (`plt_Latn`) |
| Taille | 300 M paramètres | 600 M |
| Coût | phrases 3 à 4 fois plus longues (octets) et fp32 obligatoire (les T5 débordent en fp16) : **1 h 30 à 3 h estimées** sur 2 T4 | 45 min à 1 h sur 1 T4 |
| Licence | Apache 2.0 : usage commercial possible | CC-BY-NC : non commercial |

**Anti-hallucination** : garde-fous mot et phrase identiques. Le bonus de recopie est désactivé : avec des octets,
chaque « token » (une lettre) est présent dans toutes les phrases, le bonus ne distinguerait rien.

## 1. Installation

In [ ]:
# transformers, datasets, accelerate et torch sont deja installes sur Kaggle.
!pip install -q -U sacrebleu sentencepiece

## 2. Configuration

- **`MAX_LEN = 384` octets** : une ligne (préfixe `translate ... :` compris) fait 119 octets en médiane, 278 au 99e
  centile, 336 au 99,9e et 519 au maximum. 384 garde tout sauf 3 exemples sur 13 106, retirés et non coupés.
- **`EPOCHS = 12`** est un plafond. ByT5-small apprend plus lentement que NLLB : il part de zéro pour la traduction
  et travaille sur des octets. Une époque fait ~430 pas (batch 32). Le meilleur point devrait arriver vers l'époque
  6 à 10 (estimation) ; la validation est mesurée **une fois par époque** (plus coûteuse en octets) et l'entraînement
  s'arrête après **2 époques sans progrès**. Le bloc 7 affiche l'époque réelle.
- **`LR = 5e-4` avec Adafactor**, warmup 6 %, décroissance linéaire, label smoothing 0,1. Les T5/ByT5 se fine-tunent
  avec des LR plus élevés que NLLB (1e-3 dans l'article ByT5 ; ton ancien run ByT5-base à 3e-4 apprenait bien).
- **Batch** : 8 par GPU × 2 GPU × accumulation 2 = 32. **fp32** : la T4 n'a pas de vrai bf16 et ByT5 déborde en fp16.
- **Deux GPU** : contrairement à NLLB, ici le calcul domine largement la copie des poids entre GPU,
  DataParallel fait donc gagner du temps.

In [ ]:
import os

USE_ONE_GPU = False   # ByT5 : les 2 GPU (DataParallel) font gagner du temps
if USE_ONE_GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # avant tout import de torch

CFG = {
    # --- donnees ---
    "DATASET_REPO": "nirina-doris-sylvie/betsileo-mg-v2",   # dataset Hugging Face (prive)
    "KAGGLE_INPUT": "/kaggle/input",                         # prioritaire s'il contient train.jsonl
    # --- modele ---
    "BASE_MODEL": "google/byt5-small",
    "OUTPUT_REPO": "nirina-doris-sylvie/byt5-small-betsileo-mg",
    "PUSH_TO_HUB": True,
    "LANG_CODES": {"mg": "plt_Latn", "bts": "bts_Latn"},    # bts_Latn est cree, initialise depuis plt_Latn
    # --- longueurs (tokens) ---
    "MAX_LEN": 384,          # octets
    # --- entrainement ---
    "EPOCHS": 12,
    "LR": 5e-4,
    "BATCH": 8,                # par GPU (T4 16 Go, fp32, 384 octets)
    "EFFECTIVE_BATCH": 32,     # par accumulation de gradient
    "WARMUP": 0.06,
    "WEIGHT_DECAY": 0.01,
    "LABEL_SMOOTHING": 0.1,
    "EVALS_PER_EPOCH": 1,
    "PATIENCE": 2,
    "LEXIQUE_REPEAT": 3,
    "CHANSON_REPEAT": 1,       # 2 = chansons vues 2 fois (mais 169 vers sont des traductions de Claude non relues)
    # --- generation et garde-fous ---
    "NUM_BEAMS": 4,
    "COPY_BONUS_GRID": [0.0],                         # desactive : un octet est present partout
    "WORD_MIN_GRID": [0.0, 0.1, 0.2, 0.3, 0.4, 0.5],  # proba min d'un mot avant de recopier le mot source
    "SENT_MIN_GRID": [0.0, 0.25, 0.35, 0.45, 0.55],   # proba moyenne min d'une phrase avant de la recopier
    "GUARD_TOLERANCE": 0.3,    # on prend le reglage le plus prudent a 0.3 point de chrF++ du meilleur
    "SEED": 42,
    "NUM_WORKERS": 2,
    # --- options ---
    "SMOKE_TEST": False,       # True : 20 pas sur un petit echantillon, pour verifier que tout tourne
    "FINAL_FIT": False,        # True : re-entraine sur train + validation + test (bloc 13)
}

In [ ]:
import difflib
import gc
import glob
import inspect
import json
import math
import random
import re
import time
import unicodedata
from collections import Counter

import numpy as np
import torch
import transformers
import datasets
import sacrebleu
from datasets import DatasetDict, concatenate_datasets, load_dataset
from transformers import (AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback, LogitsProcessor, LogitsProcessorList,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed)

set_seed(CFG["SEED"])
ON_KAGGLE = os.path.isdir("/kaggle/working")
WORK = "/kaggle/working" if ON_KAGGLE else os.path.abspath("work")
OUTPUT_DIR = os.path.join(WORK, "model")
os.makedirs(WORK, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FAMILY = "nllb" if "nllb" in CFG["BASE_MODEL"].lower() else "t5"
SMOKE = CFG["SMOKE_TEST"]
if SMOKE:
    CFG["PUSH_TO_HUB"] = False
if DEVICE == "cpu" and not SMOKE:
    raise RuntimeError("Pas de GPU : active Settings -> Accelerator -> GPU T4 x2, puis relance. "
                       "(Sur CPU, l'entrainement prendrait des jours.)")

print(f"torch {torch.__version__} | transformers {transformers.__version__} | "
      f"datasets {datasets.__version__} | sacrebleu {sacrebleu.__version__}")
print(f"device={DEVICE} gpus={torch.cuda.device_count()} "
      f"{torch.cuda.get_device_name(0) if DEVICE == 'cuda' else ''} | famille={FAMILY}")


def get_hf_token():
    token = os.environ.get("HF_TOKEN")
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception as e:   # on affiche la vraie raison donnee par Kaggle
            print("Lecture du secret HF_TOKEN impossible :", type(e).__name__, e)
            token = None
    if not token:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    return token


HF_TOKEN = get_hf_token()
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Hugging Face : connecte")
else:
    print("Pas de HF_TOKEN : dataset prive et envoi sur le Hub impossibles.")
    CFG["PUSH_TO_HUB"] = False

## 3. Données

Le dataset vient de `training/build_dataset.py` (dépôt betsileo-song-data). Colonne `origin` :

- `bible` : `baibolybets.json` (train), `validation-00000-of-00001.json` (validation), `test-00000-of-00001.json` (test) ;
- `chanson` : `corpus_chansons.json`, réparti **par chanson** ; validation et test ne contiennent que des
  traductions humaines (les vers traduits par Claude et pas encore relus restent en train, colonne `traducteur`) ;
- `lexique` : 85 % en train, 15 % en test (mots jamais vus) ;
- `segment` : propositions courtes découpées des versets du train.

Les scores sont donnés séparément pour la Bible et pour les chansons : les chansons sont la langue de tous les jours.

In [ ]:
def find_kaggle_data(root):
    if not os.path.isdir(root):
        return None
    hits = glob.glob(os.path.join(root, "**", "train.jsonl"), recursive=True)
    return os.path.dirname(hits[0]) if hits else None


data_dir = find_kaggle_data(CFG["KAGGLE_INPUT"])
if data_dir:
    raw = load_dataset("json", data_files={s: os.path.join(data_dir, f"{s}.jsonl")
                                           for s in ("train", "validation", "test")})
    print("Donnees : dataset Kaggle", data_dir)
else:
    raw = load_dataset(CFG["DATASET_REPO"], token=HF_TOKEN)
    print("Donnees : Hugging Face", CFG["DATASET_REPO"])

def derive(r):
    # Le dataset n'a que source / target / direction / ref : on en deduit le reste.
    mg2bts = r["direction"] == "Malagasy->Betsileo"
    ref = r["ref"]
    origin = ("lexique" if ref.startswith("LEX") else
              "chanson" if re.search(r"\.(json|txt):\d+$", ref) else "bible")
    return {"src_text": r["source"].split(": ", 1)[1] if ": " in r["source"] else r["source"],
            "src_lang": "mg" if mg2bts else "bts", "tgt_lang": "bts" if mg2bts else "mg",
            "origin": origin}


raw = raw.map(derive)


def take(ds, origin, n):
    sub = ds.filter(lambda r: r["origin"] == origin)
    return sub.select(range(min(n, len(sub))))


if SMOKE:
    raw = DatasetDict({
        "train": raw["train"].shuffle(seed=CFG["SEED"]).select(range(min(256, len(raw["train"])))),
        "validation": concatenate_datasets([take(raw["validation"], "bible", 10),
                                            take(raw["validation"], "chanson", 6)]),
        "test": concatenate_datasets([take(raw["test"], "bible", 8), take(raw["test"], "chanson", 6),
                                      take(raw["test"], "lexique", 6)]),
    })

for split in raw:
    print(f"{split:<11} {len(raw[split]):>6} lignes  {dict(Counter(raw[split]['origin']))}")
print("\nExemple :", raw["train"][0])

# Phrases (versets + chansons) pour le chrF++ ; mots du lexique a part (precision exacte).
val_sent = raw["validation"].filter(lambda r: r["origin"] != "lexique")
test_sent = raw["test"].filter(lambda r: r["origin"] != "lexique")
test_lex = raw["test"].filter(lambda r: r["origin"] == "lexique")
ORIGINS = [o for o in ("bible", "chanson") if o in set(test_sent["origin"]) | set(val_sent["origin"])]
DIRECTIONS = {"mg2bts": "Malagasy->Betsileo", "bts2mg": "Betsileo->Malagasy"}

# Meme normalisation que build_dataset.py, appliquee aux phrases tapees par l'utilisateur.
_APOS = str.maketrans({"’": "'", "‘": "'", "ʼ": "'", "`": "'", "“": '"',
                       "”": '"', "«": '"', "»": '"', " ": " "})


def normalize_text(text):
    t = unicodedata.normalize("NFC", str(text)).translate(_APOS)
    t = re.sub(r"\s+", " ", t).strip()
    return re.sub(r"\s+([,.;:!?])", r"\1", t)


def wnorm(word):
    return re.sub(r"[^\w'-]", "", word.lower())


def words(text):
    return [w for w in (wnorm(x) for x in text.split()) if w]

## 4. Baseline de recopie

Recopier la phrase source sans rien traduire obtient déjà un score, parce que les deux variétés partagent beaucoup de mots.
C'est la barre à dépasser : un modèle sous cette barre fait pire que ne rien faire.

In [ ]:
def scores(hyps, refs):
    hyps, refs = list(hyps), list(refs)
    if not refs:
        return {"chrF++": None, "BLEU": None, "n": 0}
    return {"chrF++": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 2),
            "BLEU": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2),
            "n": len(refs)}


def scores_by_direction(ds, hyps):
    out = {}
    for key, name in DIRECTIONS.items():
        idx = [i for i, d in enumerate(ds["direction"]) if d == name]
        out[key] = scores([hyps[i] for i in idx], [ds["target"][i] for i in idx])
    out["global"] = scores(hyps, ds["target"])
    return out


def scores_by_origin(ds, hyps):
    # {"tout": ..., "bible": ..., "chanson": ...}, chacun par direction.
    hyps = list(hyps)
    out = {"tout": scores_by_direction(ds, hyps)}
    for origin in ORIGINS:
        idx = [i for i, o in enumerate(ds["origin"]) if o == origin]
        if idx:
            out[origin] = scores_by_direction(ds.select(idx), [hyps[i] for i in idx])
    return out


def fmt(v):
    return "-" if v is None else f"{v:.2f}"


def print_scores(label, res):
    for part, s in res.items():
        print(f"{label:<28} {part:<8} chrF++ mg->bts {fmt(s['mg2bts']['chrF++']):>6} | bts->mg "
              f"{fmt(s['bts2mg']['chrF++']):>6} | global {fmt(s['global']['chrF++']):>6} | "
              f"BLEU {fmt(s['global']['BLEU']):>6} | n={s['global']['n']}")


COPY_BASELINE = {"validation": scores_by_origin(val_sent, val_sent["src_text"]),
                 "test": scores_by_origin(test_sent, test_sent["src_text"])}
for split, res in COPY_BASELINE.items():
    print_scores(f"recopie ({split})", res)

## 5. Tokenizer et modèle

ByT5 n'a ni vocabulaire ni code de langue : il lit les octets UTF-8 du texte. La direction lui est donnée par le
préfixe texte `translate Malagasy to Betsileo: ...` / `translate Betsileo to Malagasy: ...` (comme ton ancien
script). Les longueurs ci-dessous sont donc en octets.

In [ ]:
LANG_NAME = {"mg": "Malagasy", "bts": "Betsileo"}


def load_base(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    lang_id = {}
    if FAMILY == "nllb":
        mg_code, bts_code = CFG["LANG_CODES"]["mg"], CFG["LANG_CODES"]["bts"]
        created = bts_code not in tok.get_vocab()
        if created:
            tok.add_tokens([bts_code], special_tokens=True)
        lang_id = {"mg": tok.convert_tokens_to_ids(mg_code), "bts": tok.convert_tokens_to_ids(bts_code)}
        emb = mdl.get_input_embeddings()
        if lang_id["bts"] >= emb.weight.shape[0]:
            mdl.resize_token_embeddings(len(tok))
            emb = mdl.get_input_embeddings()
        if created:
            with torch.no_grad():
                emb.weight[lang_id["bts"]] = emb.weight[lang_id["mg"]].clone()
                out_emb = mdl.get_output_embeddings()
                if out_emb is not None and out_emb.weight.data_ptr() != emb.weight.data_ptr():
                    out_emb.weight[lang_id["bts"]] = out_emb.weight[lang_id["mg"]].clone()
            print(f"{bts_code} cree (id {lang_id['bts']}), initialise depuis {mg_code}")
    return tok, mdl, lang_id


tokenizer, model, LANG_ID = load_base(CFG["BASE_MODEL"])
print(f"Parametres : {sum(p.numel() for p in model.parameters()) / 1e6:.0f} M")


def encode_sources(texts, src_langs, tgt_langs, max_len):
    if FAMILY == "nllb":
        ids = tokenizer(list(texts), add_special_tokens=False)["input_ids"]
        return [[LANG_ID[s]] + x[: max_len - 2] + [tokenizer.eos_token_id]
                for x, s in zip(ids, src_langs)]
    prompts = [f"translate {LANG_NAME[s]} to {LANG_NAME[t]}: {x}"
               for x, s, t in zip(texts, src_langs, tgt_langs)]
    return tokenizer(prompts, max_length=max_len, truncation=True)["input_ids"]


def encode_targets(texts, tgt_langs, max_len):
    if FAMILY == "nllb":
        ids = tokenizer(list(texts), add_special_tokens=False)["input_ids"]
        return [[LANG_ID[t]] + x[: max_len - 2] + [tokenizer.eos_token_id]
                for x, t in zip(ids, tgt_langs)]
    return tokenizer(text_target=list(texts), max_length=max_len, truncation=True)["input_ids"]


tr = raw["train"]
lens = np.maximum([len(x) for x in encode_sources(tr["src_text"], tr["src_lang"], tr["tgt_lang"], 10**6)],
                  [len(x) for x in encode_targets(tr["target"], tr["tgt_lang"], 10**6)])
print("Longueur des exemples (tokens) : mediane %d | p95 %d | p99 %d | p99.9 %d | max %d"
      % (np.median(lens), np.percentile(lens, 95), np.percentile(lens, 99),
         np.percentile(lens, 99.9), lens.max()))
cover = (lens <= CFG["MAX_LEN"]).mean()
if cover < 0.99:   # autre tokenizer (ByT5 compte en octets) : on adapte au lieu de perdre des exemples
    new_len = int(math.ceil((np.percentile(lens, 99.9) + 2) / 8) * 8)
    print(f"MAX_LEN = {CFG['MAX_LEN']} ne garderait que {100 * cover:.1f} % des exemples -> MAX_LEN = {new_len}")
    CFG["MAX_LEN"] = new_len
print(f"MAX_LEN = {CFG['MAX_LEN']} garde {100 * (lens <= CFG['MAX_LEN']).mean():.2f} % des exemples")

# Verification rapide : une passe avant doit donner une perte finie.
model.to(DEVICE)
r0 = raw["train"][0]
x = torch.tensor(encode_sources([r0["src_text"]], [r0["src_lang"]], [r0["tgt_lang"]], CFG["MAX_LEN"]), device=DEVICE)
y = torch.tensor(encode_targets([r0["target"]], [r0["tgt_lang"]], CFG["MAX_LEN"]), device=DEVICE)
with torch.no_grad():
    loss0 = model(input_ids=x, attention_mask=torch.ones_like(x), labels=y).loss.item()
print(f"Perte initiale sur un exemple : {loss0:.3f}")
assert math.isfinite(loss0), "perte non finie : verifier le modele / les donnees"

## 6. Préparation des données d'entraînement

Les mots du lexique sont répétés `LEXIQUE_REPEAT` fois, les vers de chansons `CHANSON_REPEAT` fois. Les exemples plus
longs que `MAX_LEN` sont retirés (jamais coupés).

In [ ]:
def preprocess(batch, max_len):
    enc = encode_sources(batch["src_text"], batch["src_lang"], batch["tgt_lang"], max_len)
    lab = encode_targets(batch["target"], batch["tgt_lang"], max_len)
    return {"input_ids": enc, "attention_mask": [[1] * len(e) for e in enc],
            "labels": lab, "length": [len(e) for e in enc]}


def build_train_set(ds):
    parts = [ds]
    for origin, reps in (("lexique", CFG["LEXIQUE_REPEAT"]), ("chanson", CFG["CHANSON_REPEAT"])):
        sub = ds.filter(lambda r, o=origin: r["origin"] == o)
        if reps > 1 and len(sub):
            parts += [sub] * (reps - 1)
    ds = concatenate_datasets(parts).shuffle(seed=CFG["SEED"])
    tok_ds = ds.map(lambda b: preprocess(b, 10**6), batched=True, remove_columns=ds.column_names)
    n0 = len(tok_ds)
    tok_ds = tok_ds.filter(lambda r: len(r["input_ids"]) <= CFG["MAX_LEN"] and len(r["labels"]) <= CFG["MAX_LEN"])
    print(f"train : {len(tok_ds)} exemples ({n0 - len(tok_ds)} trop longs retires)")
    return tok_ds


train_tok = build_train_set(raw["train"])
val_tok = raw["validation"].map(lambda b: preprocess(b, CFG["MAX_LEN"]), batched=True,
                                remove_columns=raw["validation"].column_names)
VAL_REFS = list(raw["validation"]["target"])
print(f"validation : {len(val_tok)} exemples")

## 7. Entraînement

À chaque fin d'époque, la validation est traduite en glouton (rapide) pour mesurer le chrF++ ; deux exemples
« prédit / référence » sont affichés pour vérifier que tout va bien.

In [ ]:
def clean_decoded(text):
    text = text.replace(CFG["LANG_CODES"]["bts"], " ").replace(CFG["LANG_CODES"]["mg"], " ")
    return re.sub(r"\s+", " ", text).strip()


def compute_metrics(eval_pred):
    preds = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    preds = np.where(preds < 0, tokenizer.pad_token_id, preds)
    # Les references sont relues dans les labels du MEME lot que les predictions : le tri par
    # longueur melange l'ordre de la validation, qui n'est donc plus celui du fichier.
    labels = np.where(eval_pred.label_ids < 0, tokenizer.pad_token_id, eval_pred.label_ids)
    hyps = [clean_decoded(t) for t in tokenizer.batch_decode(preds, skip_special_tokens=True)]
    refs = [clean_decoded(t) for t in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    for h, r in list(zip(hyps, refs))[:2]:
        print(f"   predit    : {h[:90]}\n   reference : {r[:90]}")
    return {"chrf": round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 2),
            "bleu": round(sacrebleu.corpus_bleu(hyps, [refs]).score, 2)}


def make_training_args(**kw):
    # Compatible transformers 4.4x et 5.x : on adapte les noms qui ont change.
    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    if "eval_strategy" in kw and "eval_strategy" not in params:
        kw["evaluation_strategy"] = kw.pop("eval_strategy")
    if kw.pop("group_by_length", False):
        if "train_sampling_strategy" in params:
            kw["train_sampling_strategy"] = "group_by_length"
        elif "group_by_length" in params:
            kw["group_by_length"] = True
    dropped = [k for k in list(kw) if k not in params]
    for k in dropped:
        kw.pop(k)
    if dropped:
        print("Arguments ignores par cette version de transformers :", dropped)
    return Seq2SeqTrainingArguments(**kw)


class Seq2SeqCollator:
    # Avec le label smoothing, le Trainer retire les labels avant d'appeler le modele : il faut donc
    # fournir decoder_input_ids (labels decales d'un cran), ce que transformers 5 ne fait plus pour NLLB.
    def __init__(self, mdl):
        self.base = DataCollatorForSeq2Seq(tokenizer, model=mdl, label_pad_token_id=-100,
                                           pad_to_multiple_of=8 if DEVICE == "cuda" else None)
        self.start = getattr(mdl.config, "decoder_start_token_id", None)
        if self.start is None:
            self.start = mdl.generation_config.decoder_start_token_id
        self.pad = tokenizer.pad_token_id

    def __call__(self, features):
        batch = self.base(features)
        if "labels" in batch and "decoder_input_ids" not in batch:
            labels = batch["labels"]
            dec = labels.new_full(labels.shape, self.pad)
            dec[:, 1:] = labels[:, :-1]
            dec[:, 0] = self.start
            batch["decoder_input_ids"] = dec.masked_fill(dec == -100, self.pad)
        return batch


def make_trainer(mdl, args, train_ds, eval_ds, callbacks):
    collator = Seq2SeqCollator(mdl)
    kw = dict(model=mdl, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
              data_collator=collator, callbacks=callbacks)
    if eval_ds is not None:
        kw["compute_metrics"] = compute_metrics
    if "processing_class" in inspect.signature(Seq2SeqTrainer.__init__).parameters:
        kw["processing_class"] = tokenizer
    else:
        kw["tokenizer"] = tokenizer
    return Seq2SeqTrainer(**kw)


n_gpu = max(1, torch.cuda.device_count())
GRAD_ACCUM = max(1, CFG["EFFECTIVE_BATCH"] // (CFG["BATCH"] * n_gpu))
STEPS_PER_EPOCH = math.ceil(len(train_tok) / (CFG["BATCH"] * n_gpu * GRAD_ACCUM))
TOTAL_STEPS = 20 if SMOKE else STEPS_PER_EPOCH * CFG["EPOCHS"]
EVAL_STEPS = 10 if SMOKE else max(1, STEPS_PER_EPOCH // CFG["EVALS_PER_EPOCH"])
# Vrai bf16 seulement sur Ampere et plus (A100, L4...). Sur T4 / P100, PyTorch le simule
# (beaucoup plus lent) : fp16 a la place.
USE_BF16 = DEVICE == "cuda" and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = DEVICE == "cuda" and FAMILY == "nllb" and not USE_BF16   # T5/ByT5 debordent en fp16

print(f"batch effectif = {CFG['BATCH'] * n_gpu * GRAD_ACCUM} (accumulation {GRAD_ACCUM}) | "
      f"{STEPS_PER_EPOCH} pas/epoque | plafond {TOTAL_STEPS} pas | evaluation tous les {EVAL_STEPS} pas | "
      f"{'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32'}")

common = dict(
    per_device_train_batch_size=CFG["BATCH"], per_device_eval_batch_size=CFG["BATCH"] * 2,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=CFG["LR"], optim="adafactor",
    lr_scheduler_type="linear", warmup_steps=int(CFG["WARMUP"] * TOTAL_STEPS),
    weight_decay=CFG["WEIGHT_DECAY"], label_smoothing_factor=CFG["LABEL_SMOOTHING"],
    max_grad_norm=1.0, fp16=USE_FP16, bf16=USE_BF16, group_by_length=True, length_column_name="length",
    logging_steps=max(5, EVAL_STEPS // 4), logging_first_step=True, report_to="none",
    seed=CFG["SEED"], dataloader_num_workers=CFG["NUM_WORKERS"], save_only_model=True, disable_tqdm=False,
)
args = make_training_args(
    output_dir=os.path.join(WORK, "checkpoints"),
    num_train_epochs=CFG["EPOCHS"], max_steps=TOTAL_STEPS if SMOKE else -1,
    eval_strategy="steps", eval_steps=EVAL_STEPS, save_strategy="steps", save_steps=EVAL_STEPS,
    save_total_limit=2, load_best_model_at_end=True, metric_for_best_model="chrf", greater_is_better=True,
    predict_with_generate=True, generation_max_length=CFG["MAX_LEN"], generation_num_beams=1,
    **common,
)
trainer = make_trainer(model, args, train_tok, val_tok,
                       [EarlyStoppingCallback(early_stopping_patience=CFG["PATIENCE"])])

t0 = time.time()
trainer.train()
TRAIN_MINUTES = (time.time() - t0) / 60

evals = [h for h in trainer.state.log_history if "eval_chrf" in h]
print(f"\nEntrainement : {TRAIN_MINUTES:.1f} min")
print(f"{'pas':>6} {'epoque':>7} {'perte val':>10} {'chrF++':>8} {'BLEU':>7}")
for h in evals:
    print(f"{h['step']:>6} {h['epoch']:>7.2f} {h['eval_loss']:>10.4f} {h['eval_chrf']:>8.2f} {h['eval_bleu']:>7.2f}")
best = max(evals, key=lambda h: h["eval_chrf"])
BEST_EPOCH = best["epoch"]
print(f"\nMeilleur point : pas {best['step']} (epoque {BEST_EPOCH:.2f}), chrF++ validation {best['eval_chrf']:.2f} "
      f"(recopie : {COPY_BASELINE['validation']['tout']['global']['chrF++']})")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
model = trainer.model
model.eval()

## 8. Traduction avec garde-fous anti-hallucination

1. **Dictionnaire** : un mot ou une expression du lexique d'entraînement est traduit directement.
2. **Bonus de recopie** : pendant la recherche en faisceau, les tokens présents dans la phrase source reçoivent un bonus.
   Quand le modèle hésite entre inventer un mot et recopier celui de la source, la recopie gagne.
3. **Garde-fou mot à mot** : chaque mot traduit reçoit une probabilité (calculée à part, sans le bonus). Un mot sous le seuil
   qui correspond à un mot de la source est remplacé par ce mot source : s'il ne sait pas, il recopie le malgache officiel.
4. **Garde-fou phrase** : sortie vide, boucle de répétitions, longueur anormale ou phrase entière trop incertaine
   → on rend la phrase source telle quelle.

Les seuils sont réglés sur la validation au bloc 9.

In [ ]:
SPECIAL_SET = set(tokenizer.all_special_ids) | set(LANG_ID.values())
SPECIAL_IDS = sorted(SPECIAL_SET)
AMP = DEVICE == "cuda" and FAMILY == "nllb"


class SourceCopyBias(LogitsProcessor):
    # Ajoute `bonus` au score des tokens presents dans la phrase source (hors tokens speciaux).
    def __init__(self, source_ids, bonus):
        self.source_ids = source_ids
        self.bonus = bonus
        self.add = None

    def __call__(self, input_ids, scores):
        if self.add is None or self.add.shape != scores.shape:
            mask = torch.zeros((self.source_ids.shape[0], scores.shape[-1]), device=scores.device)
            mask.scatter_(1, self.source_ids.to(scores.device), 1.0)
            mask[:, [i for i in SPECIAL_IDS if i < scores.shape[-1]]] = 0.0
            reps = scores.shape[0] // mask.shape[0]          # = nombre de faisceaux
            self.add = (mask * self.bonus).repeat_interleave(reps, dim=0)
        return scores + self.add.to(scores.dtype)


def word_probs(tok_ids, tok_lp):
    # Probabilite de chaque mot = celle de son token le moins sur.
    pieces = tokenizer.convert_ids_to_tokens(tok_ids)
    groups, new_word, pending = [], True, None
    for piece, lp in zip(pieces, tok_lp):
        if piece == " " or piece == "▁":   # espace seul (ByT5, ou piece SentencePiece isolee)
            new_word, pending = True, lp if piece == "▁" else None
            continue
        if piece.startswith("▁"):
            new_word = True
        if pending is not None:
            lp, pending = min(lp, pending), None
        if new_word or not groups:
            groups.append(lp)
            new_word = False
        else:
            groups[-1] = min(groups[-1], lp)
    return [math.exp(v) for v in groups]


def pad_batch(seqs):
    width = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), width), tokenizer.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), width), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return {"input_ids": ids.to(DEVICE), "attention_mask": mask.to(DEVICE)}


@torch.no_grad()
def generate_raw(texts, direction, num_beams, copy_bonus, batch_size=16):
    src_lang, tgt_lang = ("mg", "bts") if direction == "mg2bts" else ("bts", "mg")
    results = []
    for i in range(0, len(texts), batch_size):
        chunk = list(texts[i:i + batch_size])
        batch = pad_batch(encode_sources(chunk, [src_lang] * len(chunk), [tgt_lang] * len(chunk), CFG["MAX_LEN"]))
        # Longueur max de la sortie liee a celle de l'entree : empeche le modele de partir dans une
        # longue suite inventee. (+2 : token de depart et code de langue)
        gen = dict(num_beams=num_beams, early_stopping=num_beams > 1,
                   max_length=min(CFG["MAX_LEN"], int(batch["input_ids"].shape[1] * 1.6) + 10) + 2)
        if FAMILY == "nllb":
            gen["forced_bos_token_id"] = LANG_ID[tgt_lang]
        if copy_bonus:
            gen["logits_processor"] = LogitsProcessorList([SourceCopyBias(batch["input_ids"], copy_bonus)])
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            seqs = model.generate(**batch, **gen)
            # Probabilites du modele seul (sans le bonus), recalculees sur la sortie choisie.
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                           decoder_input_ids=seqs[:, :-1]).logits
        targets = seqs[:, 1:]
        # Ligne par ligne : log_softmax sur 256 000 mots x 32 phrases d'un coup ne tient pas sur une T4.
        lp = torch.stack([torch.log_softmax(logits[j].float(), dim=-1)
                          .gather(-1, targets[j].unsqueeze(-1)).squeeze(-1) for j in range(len(chunk))])
        hyps = [clean_decoded(t) for t in tokenizer.batch_decode(seqs, skip_special_tokens=True)]
        for row, (ids, lps) in enumerate(zip(targets.tolist(), lp.tolist())):
            tok_ids, tok_lp = [], []
            for t, l in zip(ids, lps):
                if t == tokenizer.eos_token_id:
                    break
                if t in SPECIAL_SET:
                    continue
                tok_ids.append(t)
                tok_lp.append(l)
            results.append({"src": chunk[row], "hyp": hyps[row],
                            "conf": math.exp(sum(tok_lp) / max(1, len(tok_lp))),
                            "word_probs": word_probs(tok_ids, tok_lp)})
    return results


def _transplant(src_word, hyp_word):
    # Mot source, avec la ponctuation et la majuscule du mot traduit.
    core = re.sub(r"^[^\w']+|[^\w']+$", "", src_word) or src_word
    head = re.match(r"^[^\w']*", hyp_word).group(0)
    tail = re.search(r"[^\w']*$", hyp_word).group(0)
    if hyp_word[len(head):len(head) + 1].isupper():
        core = core[:1].upper() + core[1:]
    return head + core + tail


def _similar(a, b):
    return difflib.SequenceMatcher(None, wnorm(a), wnorm(b), autojunk=False).ratio()


def word_guard(src, hyp, probs, threshold):
    # Aligne la traduction sur la source ; un mot traduit incertain redevient le mot source
    # seulement quand on est sur de la correspondance.
    hw, sw = hyp.split(), src.split()
    if threshold <= 0 or len(hw) != len(probs):
        return hyp, []
    out, copied = [], []
    ops = difflib.SequenceMatcher(None, [wnorm(w) for w in sw], [wnorm(w) for w in hw], autojunk=False).get_opcodes()
    for tag, i1, i2, j1, j2 in ops:
        if tag != "replace":
            out.extend(hw[j1:j2])
        elif all(p < threshold for p in probs[j1:j2]):
            # tout le passage est incertain : on recopie le passage source entier
            out.extend(sw[i1:i2])
            copied.append((" ".join(hw[j1:j2]), " ".join(sw[i1:i2])))
        elif (i2 - i1) == (j2 - j1):
            anchored = (j2 - j1) == 1   # un seul mot change entre deux mots identiques
            for k in range(j2 - j1):
                s, h = sw[i1 + k], hw[j1 + k]
                if probs[j1 + k] < threshold and (anchored or _similar(s, h) >= 0.5):
                    out.append(_transplant(s, h))
                    copied.append((h, s))
                else:
                    out.append(h)
        else:
            out.extend(hw[j1:j2])
    return " ".join(out), copied


LOOP_RE = re.compile(r"(\b\S+(?:\s+\S+){0,3}?)(?:\s+\1){3,}")


def sentence_guard(src, hyp, conf, threshold):
    if not hyp.strip():
        return src, "sortie vide"
    if LOOP_RE.search(hyp):
        return src, "boucle de repetitions"
    ratio = len(hyp) / max(1, len(src))
    if len(src) >= 15 and not 0.4 <= ratio <= 2.5:
        return src, f"longueur anormale ({ratio:.2f})"
    if conf < threshold:
        return src, f"phrase incertaine (proba {conf:.2f})"
    return hyp, None


def apply_guards(r, word_min, sent_min):
    hyp, copied = word_guard(r["src"], r["hyp"], r["word_probs"], word_min)
    hyp, reason = sentence_guard(r["src"], hyp, r["conf"], sent_min)
    return hyp, copied, reason


def lex_key(text):
    return " ".join(words(text))


LEXICON = {"mg2bts": {}, "bts2mg": {}}
for r in raw["train"]:
    if r["origin"] == "lexique":
        d = "mg2bts" if r["direction"] == DIRECTIONS["mg2bts"] else "bts2mg"
        LEXICON[d].setdefault(lex_key(r["src_text"]), r["target"])
print(f"Lexique : {len(LEXICON['mg2bts'])} entrees mg->bts, {len(LEXICON['bts2mg'])} bts->mg")

GUARD = {"copy_bonus": 0.0, "word_min": 0.0, "sent_min": 0.0}   # regle au bloc 9


def translate(texts, direction="mg2bts", guard=True, use_lexicon=True, details=False):
    # direction : "mg2bts" (malgache officiel -> betsileo) ou "bts2mg".
    single = isinstance(texts, str)
    items = [normalize_text(t) for t in ([texts] if single else texts)]
    out = [None] * len(items)
    todo = []
    for i, t in enumerate(items):
        hit = LEXICON[direction].get(lex_key(t)) if use_lexicon else None
        if hit:
            hit = hit[:1].upper() + hit[1:] if t[:1].isupper() else hit
            out[i] = {"traduction": hit, "methode": "lexique"}
        else:
            todo.append(i)
    if todo:
        res = generate_raw([items[i] for i in todo], direction, CFG["NUM_BEAMS"],
                           GUARD["copy_bonus"] if guard else 0.0)
        for i, r in zip(todo, res):
            if guard:
                hyp, copied, reason = apply_guards(r, GUARD["word_min"], GUARD["sent_min"])
            else:
                hyp, copied, reason = r["hyp"], [], None
            out[i] = {"traduction": hyp, "methode": "recopie" if reason else "modele",
                      "raison": reason, "mots_recopies": copied, "confiance": round(r["conf"], 3)}
    if not details:
        out = [o["traduction"] for o in out]
    return out[0] if single else out

## 9. Réglage du décodage et des garde-fous (validation)

On essaie plusieurs bonus de recopie et plusieurs seuils, et on mesure le chrF++ de validation. Parmi les réglages à
moins de `GUARD_TOLERANCE` point du meilleur score, on prend **le plus prudent** (celui qui recopie le plus) :
on préfère recopier le malgache officiel que risquer un mot inventé, tant que ça ne coûte rien en score.

In [ ]:
def run_split(ds, copy_bonus, num_beams):
    res = [None] * len(ds)
    for key, name in DIRECTIONS.items():
        idx = [i for i, d in enumerate(ds["direction"]) if d == name]
        for i, r in zip(idx, generate_raw([ds["src_text"][i] for i in idx], key, num_beams, copy_bonus)):
            res[i] = r
    return res


def chrf(hyps, refs):
    return sacrebleu.corpus_chrf(list(hyps), [list(refs)], word_order=2).score


def safest(grid_scores, tol):
    # grid_scores : [(valeur, score)] ; plus grande valeur a moins de tol du meilleur score.
    top = max(s for _, s in grid_scores)
    return max(v for v, s in grid_scores if s >= top - tol)


tol = CFG["GUARD_TOLERANCE"]
refs = list(val_sent["target"])          # versets + chansons de validation
bonus_runs = {b: run_split(val_sent, b, CFG["NUM_BEAMS"]) for b in CFG["COPY_BONUS_GRID"]}
bonus_scores = [(b, chrf([r["hyp"] for r in res], refs)) for b, res in bonus_runs.items()]
GUARD["copy_bonus"] = safest(bonus_scores, tol)
val_res = bonus_runs[GUARD["copy_bonus"]]

word_scores = [(w, chrf([word_guard(r["src"], r["hyp"], r["word_probs"], w)[0] for r in val_res], refs))
               for w in CFG["WORD_MIN_GRID"]]
GUARD["word_min"] = safest(word_scores, tol)
sent_scores = [(s, chrf([apply_guards(r, GUARD["word_min"], s)[0] for r in val_res], refs))
               for s in CFG["SENT_MIN_GRID"]]
GUARD["sent_min"] = safest(sent_scores, tol)

print("Bonus de recopie :", [(b, round(s, 2)) for b, s in bonus_scores])
print("Seuil mot        :", [(w, round(s, 2)) for w, s in word_scores])
print("Seuil phrase     :", [(s, round(v, 2)) for s, v in sent_scores])
print("\nReglage retenu :", GUARD)

## 10. Évaluation finale sur le test

Quatre systèmes comparés sur les mêmes phrases :
la recopie (baseline), le modèle brut, le modèle avec bonus de recopie, et le modèle avec tous les garde-fous.
Chaque score est donné pour tout le test, pour la Bible et pour les chansons (langue de tous les jours, traduites par toi).

- **mots inventés** : part des mots produits qui n'existent ni dans la phrase source ni dans aucun texte du train
  pour cette langue. C'est la mesure d'hallucination la plus directe pour deux variétés aussi proches.
- **lexique (test)** : mots jamais vus à l'entraînement, précision exacte.

In [ ]:
TRAIN_VOCAB = {"mg": set(), "bts": set()}
for r in raw["train"]:
    TRAIN_VOCAB[r["src_lang"]].update(words(r["src_text"]))
    TRAIN_VOCAB[r["tgt_lang"]].update(words(r["target"]))


def invented_rate(ds, hyps):
    total = invented = 0
    for src, hyp, lang in zip(ds["src_text"], hyps, ds["tgt_lang"]):
        known = TRAIN_VOCAB[lang] | set(words(src))
        ws = words(hyp)
        total += len(ws)
        invented += sum(1 for w in ws if w not in known)
    return round(100 * invented / max(1, total), 2)


raw_res = run_split(test_sent, 0.0, CFG["NUM_BEAMS"])
bias_res = run_split(test_sent, GUARD["copy_bonus"], CFG["NUM_BEAMS"]) if GUARD["copy_bonus"] else raw_res
guarded = [apply_guards(r, GUARD["word_min"], GUARD["sent_min"]) for r in bias_res]
systems = {
    "recopie (baseline)": list(test_sent["src_text"]),
    "modele brut": [r["hyp"] for r in raw_res],
    "modele + bonus de recopie": [r["hyp"] for r in bias_res],
    "modele + garde-fous": [g[0] for g in guarded],
}
RESULTS = {"test": {}, "config": CFG, "guard": GUARD, "best_epoch": BEST_EPOCH, "train_minutes": TRAIN_MINUTES}
for name, hyps in systems.items():
    s = scores_by_origin(test_sent, hyps)
    s["mots_inventes_pct"] = invented_rate(test_sent, hyps)
    RESULTS["test"][name] = s
    print_scores(name, {k: v for k, v in s.items() if k != "mots_inventes_pct"})
    print(f"{'':<28} mots inventes : {s['mots_inventes_pct']} %\n")

n_words_copied = sum(len(g[1]) for g in guarded)
n_sent_copied = sum(1 for g in guarded if g[2])
print(f"\nGarde-fous sur le test : {n_words_copied} mots recopies, {n_sent_copied}/{len(guarded)} phrases recopiees")
RESULTS["test"]["garde_fous"] = {"mots_recopies": n_words_copied, "phrases_recopiees": n_sent_copied}

# Mots du lexique jamais vus a l'entrainement
lex_hyps = [None] * len(test_lex)
for key, name in DIRECTIONS.items():
    idx = [i for i, d in enumerate(test_lex["direction"]) if d == name]
    for i, h in zip(idx, translate([test_lex["src_text"][i] for i in idx], key)):
        lex_hyps[i] = h
exact = [lex_key(h) == lex_key(t) for h, t in zip(lex_hyps, test_lex["target"])]
copied = [lex_key(h) == lex_key(s) for h, s in zip(lex_hyps, test_lex["src_text"])]
RESULTS["test"]["lexique"] = {"precision_exacte_pct": round(100 * sum(exact) / max(1, len(exact)), 1),
                              "recopie_pct": round(100 * sum(copied) / max(1, len(copied)), 1),
                              "n": len(exact)}
print("Lexique (mots jamais vus) :", RESULTS["test"]["lexique"])
for s, h, t in list(zip(test_lex["src_text"], lex_hyps, test_lex["target"]))[:10]:
    print(f"   {s:<18} -> {h:<18} (attendu : {t})")

with open(os.path.join(WORK, "predictions_test.tsv"), "w", encoding="utf-8") as f:
    f.write("origine\tref\tdirection\tsource\treference\tmodele_brut\tmodele_garde_fous\traison_recopie\n")
    for i in range(len(test_sent)):
        f.write("\t".join([test_sent["origin"][i], test_sent["ref"][i], test_sent["direction"][i],
                           test_sent["src_text"][i], test_sent["target"][i], raw_res[i]["hyp"],
                           guarded[i][0], guarded[i][2] or ""]) + "\n")
with open(os.path.join(WORK, "resultats.json"), "w", encoding="utf-8") as f:
    json.dump(RESULTS, f, ensure_ascii=False, indent=2, default=str)
print("\nEcrit :", os.path.join(WORK, "resultats.json"), "et predictions_test.tsv")

## 11. Exemples

In [ ]:
examples = {
    "mg2bts": ["Tsy fantatro izay nolazainy.", "Aiza ianao no mipetraka?", "Mandehana any an-tsena ary vidio ny vary.",
               "olona", "rehetra", "zavatra"],
    "bts2mg": ["Tsa fantatsiko izay notarogniny.", "Aia agnao ro mipetraka?", "ona", "raha aby", "tsa"],
}
for direction, texts in examples.items():
    print(f"--- {direction} ---")
    for t, r in zip(texts, translate(texts, direction, details=True)):
        extra = f" | recopie : {r['raison']}" if r.get("raison") else ""
        extra += f" | mots recopies : {r['mots_recopies']}" if r.get("mots_recopies") else ""
        print(f"{t:<45} -> {r['traduction']}   [{r['methode']}{extra}]")

## 12. Sauvegarde et envoi sur Hugging Face

Le dossier du modèle contient aussi `inference_config.json` (codes de langue, réglage des garde-fous, lexique) pour
reproduire exactement `translate()` dans le backend.

In [ ]:
def save_inference_assets(folder):
    cfg = {"family": FAMILY, "base_model": CFG["BASE_MODEL"], "lang_codes": CFG["LANG_CODES"],
           "max_len": CFG["MAX_LEN"], "num_beams": CFG["NUM_BEAMS"], "guard": GUARD,
           "lexicon": LEXICON, "normalisation": "NFC, apostrophes -> ', guillemets -> \""}
    with open(os.path.join(folder, "inference_config.json"), "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=1)
    t = RESULTS["test"]
    if FAMILY == "nllb":
        licence = "cc-by-nc-4.0"
        usage = (f"Codes de langue : malgache officiel `{CFG['LANG_CODES']['mg']}`, betsileo "
                 f"`{CFG['LANG_CODES']['bts']}` (ajoute). Forcer le premier token de sortie sur le code "
                 "de la langue cible (`forced_bos_token_id`).")
    else:
        licence = "apache-2.0"
        usage = ("Direction donnee par le prefixe texte : `translate Malagasy to Betsileo: ...` ou "
                 "`translate Betsileo to Malagasy: ...`.")
    card = f'''---
language: [mg]
license: {licence}
base_model: {CFG["BASE_MODEL"]}
tags: [translation, betsileo, malagasy]
---
# {CFG["BASE_MODEL"].split("/")[-1]} : betsileo <-> malgache officiel

Fine-tuning de `{CFG["BASE_MODEL"]}` sur `{CFG["DATASET_REPO"]}`.
{usage}

| Test (chrF++ global) | tout | bible | chansons | mots inventes |
|---|---|---|---|---|
''' + "\n".join(f"| {k} | " + " | ".join(fmt(v[o]["global"]["chrF++"]) if o in v else "-"
                                        for o in ("tout", "bible", "chanson"))
                + f" | {v['mots_inventes_pct']} % |" for k, v in t.items() if isinstance(v, dict) and "tout" in v) + f'''

Garde-fous anti-hallucination : voir `inference_config.json`.
'''
    with open(os.path.join(folder, "README.md"), "w", encoding="utf-8") as f:
        f.write(card)


def push_folder(folder, repo_id):
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id, private=True, exist_ok=True)
    api.upload_folder(folder_path=folder, repo_id=repo_id, commit_message="fine-tuning " + CFG["BASE_MODEL"] + " betsileo <-> malgache")
    print("Envoye : https://huggingface.co/" + repo_id)


save_inference_assets(OUTPUT_DIR)
if CFG["PUSH_TO_HUB"]:
    push_folder(OUTPUT_DIR, CFG["OUTPUT_REPO"])
else:
    print("Envoi sur le Hub desactive. Modele dans", OUTPUT_DIR)

## 13. (Optionnel) Modèle final sur toutes les données

Une fois le score mesuré honnêtement ci-dessus, on peut ré-entraîner sur **train + validation + test** (les 4 fichiers
regroupés) pendant le nombre d'époques qui a donné le meilleur point. Ce modèle n'a plus de test pour le mesurer : c'est
celui à mettre en production, pas celui à citer dans le mémoire. Il est envoyé dans un dépôt séparé (`OUTPUT_REPO-final`).
Coût : à peu près le temps du bloc 7.

In [ ]:
if CFG["FINAL_FIT"]:
    del trainer, model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    tokenizer, model, LANG_ID = load_base(CFG["BASE_MODEL"])
    all_data = concatenate_datasets([raw["train"], raw["validation"], raw["test"]])
    all_tok = build_train_set(all_data)
    epochs = max(1.0, math.ceil(BEST_EPOCH * 2) / 2)
    steps = math.ceil(len(all_tok) / (CFG["BATCH"] * n_gpu * GRAD_ACCUM) * epochs)
    final_args = make_training_args(output_dir=os.path.join(WORK, "checkpoints_final"),
                                    num_train_epochs=epochs, eval_strategy="no", save_strategy="no",
                                    **{**common, "warmup_steps": int(CFG["WARMUP"] * steps)})
    final_trainer = make_trainer(model, final_args, all_tok, None, [])
    final_trainer.train()
    final_dir = os.path.join(WORK, "model_final")
    final_trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)
    model = final_trainer.model
    model.eval()
    for r in all_data:
        if r["origin"] == "lexique":
            d = "mg2bts" if r["direction"] == DIRECTIONS["mg2bts"] else "bts2mg"
            LEXICON[d].setdefault(lex_key(r["src_text"]), r["target"])
    save_inference_assets(final_dir)
    if CFG["PUSH_TO_HUB"]:
        push_folder(final_dir, CFG["OUTPUT_REPO"] + "-final")
    print(f"Modele final : {epochs} epoques sur {len(all_tok)} exemples -> {final_dir}")
else:
    print("FINAL_FIT = False : bloc ignore.")